# Module 3: RAG Systems Engineering - Hybrid Search & GraphRAG

This notebook is refactored into smaller implementation blocks so each stage is easy to run and debug independently.


## 1. Setup: Dependencies and Runtime Configuration

First load all dependencies used throughout ingestion and retrieval.


In [ ]:
import math
import json
import os
from typing import List, Dict, Any, Tuple, Optional
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict

import requests
import networkx as nx
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

print("✓ Imports successful")


### Configuration Constants

Set service endpoints for OpenAI embeddings and Qdrant vector storage.


In [ ]:
# OpenAI embeddings configuration
OPENAI_API_URL = "https://api.openai.com/v1"
OPENAI_EMBED_MODEL = "text-embedding-3-small"

# Optional: load OPENAI_API_KEY from repo-root .env
root_env = Path("..") / ".env"
if root_env.exists() and "OPENAI_API_KEY" not in os.environ:
    for line in root_env.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        if key.strip() == "OPENAI_API_KEY":
            os.environ["OPENAI_API_KEY"] = value.strip().strip("'"")
            break

# Qdrant connection
QDRANT_URL = "http://localhost:6333"

print("✓ Runtime configuration set")
print("✓ OPENAI_API_KEY loaded" if os.getenv("OPENAI_API_KEY") else "⚠ OPENAI_API_KEY not found")


## 2. Document Ingestion with Hierarchical Chunking

Define the core chunk schema that preserves section structure and metadata.


In [ ]:
@dataclass
class DocumentChunk:
    """Represents a chunk of text from the policy document."""

    chunk_id: str
    text: str
    section_number: str  # e.g., "4.1", "9.2.1"
    section_title: str   # e.g., "Taxis and Rideshares"
    parent_section: Optional[str]  # e.g., "4" for "4.1"
    page_number: int
    hierarchy_path: List[str]  # e.g., ["Section 4", "4.1"]
    chunk_type: str  # "section", "subsection", "paragraph"

    def to_dict(self) -> Dict[str, Any]:
        return {
            "chunk_id": self.chunk_id,
            "text": self.text,
            "section_number": self.section_number,
            "section_title": self.section_title,
            "parent_section": self.parent_section,
            "page_number": self.page_number,
            "hierarchy_path": self.hierarchy_path,
            "chunk_type": self.chunk_type,
        }

print("✓ DocumentChunk defined")


### Ingestion Pipeline

The pipeline loads PDF pages, detects sections/subsections, and emits hierarchical chunks.


In [ ]:
class DocumentIngestionPipeline:
    """Ingests PDF documents with hierarchical chunking."""

    def __init__(self, pdf_path: str):
        self.pdf_path = Path(pdf_path)
        self.chunks: List[DocumentChunk] = []

    def load_pdf(self) -> List[Tuple[int, str]]:
        """Load PDF and extract text by page."""
        reader = PdfReader(self.pdf_path)
        pages = []

        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text()
            pages.append((page_num, text))

        print(f"✓ Loaded {len(pages)} pages from {self.pdf_path.name}")
        return pages

    def detect_sections(self, pages: List[Tuple[int, str]]) -> List[DocumentChunk]:
        """Detect hierarchical sections and create chunks."""
        import re

        chunks = []
        current_section = None
        current_subsection = None
        chunk_counter = 0

        # Patterns for section detection
        section_pattern = re.compile(r'^Section (\d+):\s*(.+)$', re.MULTILINE)
        subsection_pattern = re.compile(r'^(\d+\.\d+)\s+(.+)$', re.MULTILINE)
        subsubsection_pattern = re.compile(r'^(\d+\.\d+\.\d+)\s+(.+)$', re.MULTILINE)

        for page_num, page_text in pages:
            # Skip title page and TOC (first 3 pages)
            if page_num <= 3:
                continue

            # Split into paragraphs
            paragraphs = [p.strip() for p in page_text.split('\n\n') if p.strip()]

            for para in paragraphs:
                # Check for main section
                section_match = section_pattern.search(para)
                if section_match:
                    section_num = section_match.group(1)
                    section_title = section_match.group(2).strip()
                    current_section = section_num
                    current_subsection = None

                    chunk_counter += 1
                    chunks.append(DocumentChunk(
                        chunk_id=f"chunk_{chunk_counter}",
                        text=para,
                        section_number=section_num,
                        section_title=section_title,
                        parent_section=None,
                        page_number=page_num,
                        hierarchy_path=[f"Section {section_num}"],
                        chunk_type="section"
                    ))
                    continue

                # Check for subsection (e.g., 4.1)
                subsection_match = subsection_pattern.search(para)
                if subsection_match and current_section:
                    subsection_num = subsection_match.group(1)
                    subsection_title = subsection_match.group(2).strip()
                    current_subsection = subsection_num

                    chunk_counter += 1
                    chunks.append(DocumentChunk(
                        chunk_id=f"chunk_{chunk_counter}",
                        text=para,
                        section_number=subsection_num,
                        section_title=subsection_title,
                        parent_section=current_section,
                        page_number=page_num,
                        hierarchy_path=[f"Section {current_section}", subsection_num],
                        chunk_type="subsection"
                    ))
                    continue

                # Check for sub-subsection (e.g., 4.1.1)
                subsubsection_match = subsubsection_pattern.search(para)
                if subsubsection_match and current_subsection:
                    subsubsection_num = subsubsection_match.group(1)
                    subsubsection_title = subsubsection_match.group(2).strip()

                    chunk_counter += 1
                    chunks.append(DocumentChunk(
                        chunk_id=f"chunk_{chunk_counter}",
                        text=para,
                        section_number=subsubsection_num,
                        section_title=subsubsection_title,
                        parent_section=current_subsection,
                        page_number=page_num,
                        hierarchy_path=[f"Section {current_section}", current_subsection, subsubsection_num],
                        chunk_type="subsubsection"
                    ))
                    continue

                # Regular paragraph - attach to current subsection or section
                if len(para) > 50:  # Only chunk substantial paragraphs
                    chunk_counter += 1
                    parent = current_subsection or current_section
                    hierarchy = []
                    if current_section:
                        hierarchy.append(f"Section {current_section}")
                    if current_subsection:
                        hierarchy.append(current_subsection)

                    chunks.append(DocumentChunk(
                        chunk_id=f"chunk_{chunk_counter}",
                        text=para,
                        section_number=parent or "0",
                        section_title="",
                        parent_section=parent,
                        page_number=page_num,
                        hierarchy_path=hierarchy,
                        chunk_type="paragraph"
                    ))

        print(f"✓ Created {len(chunks)} hierarchical chunks")
        return chunks

    def ingest(self) -> List[DocumentChunk]:
        """Run the full ingestion pipeline."""
        pages = self.load_pdf()
        self.chunks = self.detect_sections(pages)
        return self.chunks

print("✓ DocumentIngestionPipeline defined")


### Run Ingestion

Ingest the sample policy and inspect a few chunk examples.


In [ ]:
# Ingest the policy document
pipeline = DocumentIngestionPipeline("sample_expense_policy.pdf")
chunks = pipeline.ingest()

# Show some example chunks
print("\n=== Example Chunks ===")
for i, chunk in enumerate(chunks[:5]):
    print(f"\nChunk {i+1}:")
    print(f"  ID: {chunk.chunk_id}")
    print(f"  Section: {chunk.section_number} - {chunk.section_title}")
    print(f"  Parent: {chunk.parent_section}")
    print(f"  Page: {chunk.page_number}")
    print(f"  Hierarchy: {' > '.join(chunk.hierarchy_path)}")
    print(f"  Type: {chunk.chunk_type}")
    print(f"  Text preview: {chunk.text[:100]}...")

print(f"\n✓ Total chunks created: {len(chunks)}")


## 3. Dense Vector Retrieval (Semantic Search)

Dense retrieval captures semantic similarity using embeddings and Qdrant.


In [ ]:
class DenseVectorRetriever:
    """Dense vector retrieval using Qdrant and OpenAI embeddings."""

    def __init__(
        self,
        collection_name: str = "expense_policy",
        embed_model: str = OPENAI_EMBED_MODEL,
        qdrant_url: str = QDRANT_URL,
    ):
        self.collection_name = collection_name
        self.embed_model = embed_model
        self.client = QdrantClient(url=qdrant_url)
        self.embedding_dim = 1536  # text-embedding-3-small dimension

    def _embed(self, text: str) -> List[float]:
        """Generate embedding for text using OpenAI."""
        api_key = os.getenv("OPENAI_API_KEY")
        if not api_key:
            raise ValueError("OPENAI_API_KEY is not set.")

        response = requests.post(
            f"{OPENAI_API_URL}/embeddings",
            headers={"Authorization": f"Bearer {api_key}"},
            json={"model": self.embed_model, "input": text},
            timeout=30,
        )
        response.raise_for_status()
        return response.json()["data"][0]["embedding"]

    def create_collection(self):
        """Create Qdrant collection for policy chunks."""
        try:
            self.client.delete_collection(self.collection_name)
        except:
            pass

        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=VectorParams(
                size=self.embedding_dim,
                distance=Distance.COSINE
            )
        )
        print(f"✓ Created Qdrant collection: {self.collection_name}")

    def index_chunks(self, chunks: List[DocumentChunk]):
        """Index document chunks in Qdrant."""
        print(f"Indexing {len(chunks)} chunks...")

        points = []
        for i, chunk in enumerate(chunks):
            if i % 10 == 0:
                print(f"  Embedding chunk {i+1}/{len(chunks)}...")

            embedding = self._embed(chunk.text)
            point = PointStruct(
                id=i,
                vector=embedding,
                payload=chunk.to_dict()
            )
            points.append(point)

        self.client.upsert(
            collection_name=self.collection_name,
            points=points
        )
        print(f"✓ Indexed {len(points)} chunks in Qdrant")

    def search(self, query: str, top_k: int = 5) -> List[Tuple[DocumentChunk, float]]:
        """Semantic search for relevant chunks."""
        query_embedding = self._embed(query)

        results = self.client.query_points(
            collection_name=self.collection_name,
            query=query_embedding,
            limit=top_k
        ).points

        retrieved = []
        for result in results:
            chunk = DocumentChunk(**result.payload)
            score = result.score
            retrieved.append((chunk, score))

        return retrieved

print("✓ DenseVectorRetriever defined")


### Build Dense Index

Create the vector collection and upload chunk embeddings.


In [ ]:
# Create dense retriever and index chunks
dense_retriever = DenseVectorRetriever()
dense_retriever.create_collection()
dense_retriever.index_chunks(chunks)

print("\n✓ Dense vector indexing complete")


### Test Dense Search

Run semantic queries and inspect top results.


In [ ]:
# Test semantic search
test_queries = [
    "What is the meal limit for individual employees?",
    "Can I expense sustenance while traveling?",
    "What are the rules for client dinners?",
]

print("=== Dense Vector Search Results ===")
for query in test_queries:
    print(f"\nQuery: {query}")
    results = dense_retriever.search(query, top_k=3)

    for i, (chunk, score) in enumerate(results, 1):
        print(f"\n  Result {i} (score: {score:.3f}):")
        print(f"    Section: {chunk.section_number} - {chunk.section_title}")
        print(f"    Page: {chunk.page_number}")
        print(f"    Text: {chunk.text[:150]}...")


## 4. Sparse BM25 Retrieval (Keyword Search)

Sparse retrieval is effective for exact codes, amounts, and phrase matching.


In [ ]:
class SparseBM25Retriever:
    """Sparse keyword retrieval using BM25."""

    def __init__(self):
        self.chunks: List[DocumentChunk] = []
        self.bm25: Optional[BM25Okapi] = None
        self.tokenized_corpus: List[List[str]] = []

    def _tokenize(self, text: str) -> List[str]:
        """Simple tokenization: lowercase and split on whitespace."""
        return text.lower().split()

    def index_chunks(self, chunks: List[DocumentChunk]):
        """Build BM25 index over chunks."""
        self.chunks = chunks
        self.tokenized_corpus = [self._tokenize(chunk.text) for chunk in chunks]
        self.bm25 = BM25Okapi(self.tokenized_corpus)
        print(f"✓ Built BM25 index over {len(chunks)} chunks")

    def search(self, query: str, top_k: int = 5) -> List[Tuple[DocumentChunk, float]]:
        """Keyword search for relevant chunks."""
        if not self.bm25:
            raise ValueError("BM25 index not built. Call index_chunks() first.")

        tokenized_query = self._tokenize(query)
        scores = self.bm25.get_scores(tokenized_query)
        top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]

        results = [(self.chunks[i], scores[i]) for i in top_indices]
        return results

print("✓ SparseBM25Retriever defined")


### Build and Test BM25 Index

Run keyword-heavy queries including policy codes and dollar limits.


In [ ]:
# Create BM25 retriever and index chunks
bm25_retriever = SparseBM25Retriever()
bm25_retriever.index_chunks(chunks)

# Test keyword search
test_queries = [
    "PROJ-2024-001",
    "$150 client dinner",
    "VP first class",
]

print("\n=== BM25 Keyword Search Results ===")
for query in test_queries:
    print(f"\nQuery: {query}")
    results = bm25_retriever.search(query, top_k=3)

    for i, (chunk, score) in enumerate(results, 1):
        print(f"\n  Result {i} (score: {score:.3f}):")
        print(f"    Section: {chunk.section_number} - {chunk.section_title}")
        print(f"    Page: {chunk.page_number}")
        print(f"    Text: {chunk.text[:150]}...")


## 5. Hybrid Search: Dense + Sparse Fusion

Hybrid retrieval combines semantic recall and exact-match precision using Reciprocal Rank Fusion (RRF).


In [ ]:
class HybridRetriever:
    """Hybrid retrieval combining dense vector and sparse BM25 search."""

    def __init__(
        self,
        dense_retriever: DenseVectorRetriever,
        sparse_retriever: SparseBM25Retriever,
        dense_weight: float = 0.5,
        sparse_weight: float = 0.5,
    ):
        self.dense_retriever = dense_retriever
        self.sparse_retriever = sparse_retriever
        self.dense_weight = dense_weight
        self.sparse_weight = sparse_weight

    def reciprocal_rank_fusion(
        self,
        dense_results: List[Tuple[DocumentChunk, float]],
        sparse_results: List[Tuple[DocumentChunk, float]],
        k: int = 60,
    ) -> List[Tuple[DocumentChunk, float]]:
        """Merge results using Reciprocal Rank Fusion.

        RRF score = sum(1 / (k + rank)) for each retriever where doc appears.
        """
        scores = defaultdict(float)
        chunk_map = {}

        for rank, (chunk, _) in enumerate(dense_results, start=1):
            rrf_score = self.dense_weight / (k + rank)
            scores[chunk.chunk_id] += rrf_score
            chunk_map[chunk.chunk_id] = chunk

        for rank, (chunk, _) in enumerate(sparse_results, start=1):
            rrf_score = self.sparse_weight / (k + rank)
            scores[chunk.chunk_id] += rrf_score
            chunk_map[chunk.chunk_id] = chunk

        sorted_chunks = sorted(
            scores.items(),
            key=lambda x: x[1],
            reverse=True
        )

        return [(chunk_map[chunk_id], score) for chunk_id, score in sorted_chunks]

    def search(self, query: str, top_k: int = 5) -> List[Tuple[DocumentChunk, float]]:
        """Hybrid search combining dense and sparse retrieval."""
        dense_results = self.dense_retriever.search(query, top_k=top_k * 2)
        sparse_results = self.sparse_retriever.search(query, top_k=top_k * 2)
        hybrid_results = self.reciprocal_rank_fusion(dense_results, sparse_results)
        return hybrid_results[:top_k]

print("✓ HybridRetriever defined")


### Test Hybrid Search

Run mixed semantic + keyword queries and inspect fused ranking.


In [ ]:
# Create hybrid retriever
hybrid_retriever = HybridRetriever(
    dense_retriever=dense_retriever,
    sparse_retriever=bm25_retriever,
    dense_weight=0.6,  # Slightly favor semantic search
    sparse_weight=0.4,
)

# Test hybrid search
test_queries = [
    "What is the meal allowance for sustenance?",  # Semantic query
    "PROJ-2024-002 client development",  # Keyword query
    "Can VP fly first class?",  # Mixed query
]

print("=== Hybrid Search Results ===")
for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")

    results = hybrid_retriever.search(query, top_k=3)

    for i, (chunk, score) in enumerate(results, 1):
        print(f"\nResult {i} (RRF score: {score:.4f}):")
        print(f"  Section: {chunk.section_number} - {chunk.section_title}")
        print(f"  Page: {chunk.page_number}")
        print(f"  Hierarchy: {' > '.join(chunk.hierarchy_path)}")
        print(f"  Text: {chunk.text[:200]}...")


## Next Step

If you want to continue, add GraphRAG entity/relationship extraction and policy exception traversal as the next section.
